# Descomposición de Series de Tiempo
### Clásica vs. STL

`Fase 1 · Video 4` — serie **Forecasting de Ventas de la A a la Z**

En el **Video 3** estabilizamos la serie. Ahora, en lugar de eliminar sus componentes, los
**separamos y entendemos** — tendencia, estacionalidad y residual, cada uno con su lectura de negocio.
Es la base de la desestacionalización que usaremos en el ACF del Video 5.

---
## Nota conceptual: Estacionariedad (V3) vs. Descomposición (V4)

Es la confusión más común de esta fase. Son **dos lentes distintos sobre los mismos componentes**, no dos pasos de la misma operación:

| | **Estacionariedad (V3)** | **Descomposición (V4)** |
|---|---|---|
| **Objetivo** | *Preparar para modelar* (ARIMA exige estacionariedad) | *Entender e interpretar* los componentes |
| **Tendencia** | La **elimina** por diferenciación $(1-B)$ | La **estima y separa** (no la borra) |
| **Estacionalidad** | La elimina por diferenciación estacional $(1-B^s)$ | La **aísla** (para desestacionalizar / re-estacionalizar) |
| **Logaritmo** | Estabiliza **varianza** (multiplicativo→aditivo) | Igual: linealiza para poder separar |

> ⚠️ **El logaritmo NO elimina tendencia ni estacionalidad** — solo estabiliza la varianza. En el mundo de la estacionariedad, la estacionalidad se quita con diferenciación **estacional** $(1-B^s)$, nunca con log ni con diferenciación simple.


---
## 1. Librerías y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_percentage_error
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
money_fmt = FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')
print('Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
df_raw   = pd.read_csv(csv_path, nrows=2)
date_col = next(c for c in df_raw.columns if 'date' in c.strip().lower())
df = pd.read_csv(csv_path, parse_dates=[date_col])
df.rename(columns={date_col: 'date'}, inplace=True)
df.sort_values('date', inplace=True)

serie = df.groupby('date')['revenue'].sum().resample('W-MON').sum()
serie.name = 'revenue'

print(f'✅ {len(df):,} filas | {df["date"].min().date()} → {df["date"].max().date()}')
print(f'   Serie semanal: {len(serie)} observaciones | Media: ${serie.mean():,.0f}')

---
## 2. Diagnóstico: ¿Aditivo o Multiplicativo?

La elección del modelo no es visual — es cuantificable. La clave está en cómo se comporta
la **amplitud estacional** a medida que sube el nivel de la serie:

| Señal | Interpretación | Modelo |
|---|---|---|
| La desviación estándar **crece con la media** | La amplitud escala con el nivel | **Multiplicativo** → log → STL |
| La desviación estándar es **estable** aunque suba la media | Amplitud constante en $ | **Aditivo** → STL directo |

El indicador más robusto es la **correlación entre la media móvil y la desviación móvil**:
si al subir el nivel sube también la amplitud (corr ≳ 0.6), la serie es multiplicativa.
(El coeficiente de variación por año se muestra como apoyo, pero es más frágil: mezcla el
crecimiento intra-anual con la amplitud estacional.)

STL es inherentemente aditivo. Para series multiplicativas aplicamos `log` primero:
`log(T × S × R) = log(T) + log(S) + log(R)` → aditivo → STL → `expm1` para volver a escala original.

In [ ]:
# Señal principal: ¿la amplitud (std móvil) crece con el nivel (media móvil)?
roll_mean = serie.rolling(12).mean()
roll_std  = serie.rolling(12).std()
corr_ms   = pd.concat([roll_mean, roll_std], axis=1).dropna().corr().iloc[0, 1]

# Apoyo: CV por año, filtrando años incompletos (< 50 semanas) para no sesgar
weeks_per_year = serie.groupby(serie.index.year).count()
full_years     = weeks_per_year[weeks_per_year >= 50].index
cv_by_year = (
    serie[serie.index.year.isin(full_years)]
    .groupby(serie[serie.index.year.isin(full_years)].index.year)
    .apply(lambda x: x.std() / x.mean())
    .rename('cv')
)

is_mult    = corr_ms > 0.6
model_type = 'MULTIPLICATIVO' if is_mult else 'ADITIVO'
transform  = 'log1p → STL → expm1' if is_mult else 'STL directo'

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(serie.index, serie.values, color=BLUE, linewidth=1.2)
axes[0].yaxis.set_major_formatter(money_fmt)
axes[0].set_title('Serie Completa — ¿Los picos crecen?', fontsize=12, fontweight='bold')

colors_yr = [BLUE, GREEN, ORANGE, PURPLE, RED]
for yr, color in zip(sorted(serie.index.year.unique()), colors_yr):
    s_yr = serie[serie.index.year == yr]
    axes[1].plot(range(len(s_yr)), s_yr.values, color=color, linewidth=2, label=str(yr), alpha=0.85)
axes[1].yaxis.set_major_formatter(money_fmt)
axes[1].set_title('Años Superpuestos — ¿Proporcional o constante?', fontsize=12, fontweight='bold')
axes[1].legend(title='Año')

fig.suptitle('Diagnóstico Visual: Aditivo vs Multiplicativo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

fig2, ax = plt.subplots(figsize=(8, 4))
ax.scatter(roll_mean, roll_std, s=16, color=BLUE, alpha=0.5, edgecolor='white', linewidth=0.4)
ax.xaxis.set_major_formatter(money_fmt)
ax.yaxis.set_major_formatter(money_fmt)
ax.set_xlabel('Media móvil (nivel)')
ax.set_ylabel('Desv. estándar móvil (amplitud)')
ax.set_title(f'¿La amplitud escala con el nivel?   corr = {corr_ms:+.2f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('─' * 50)
print('  DECISIÓN: Tipo de Modelo')
print('─' * 50)
print(f'  corr(media móvil, std móvil): {corr_ms:+.3f}   (umbral > 0.60 → multiplicativo)')
for yr, val in cv_by_year.items():
    print(f'  CV {yr} (apoyo): {val:.4f}')
print(f'  → Modelo elegido : {model_type}')
print(f'  → Transformación : {transform}')
print('─' * 50)

---
## 3. Transformación y Descomposición

Aplicamos el flujo correspondiente al tipo de modelo detectado en la sección anterior.  
Corremos ambos métodos (Clásico y STL) para poder compararlos en la sección 6.

In [ ]:
serie_work = np.log1p(serie) if is_mult else serie
label_transform = 'log(1 + serie)' if is_mult else 'serie original'
print(f'✅ Trabajando sobre: {label_transform}')

In [ ]:
decomp_cls = seasonal_decompose(serie_work, model='additive', period=52)

def plot_decomposition(result, title, escala_orig=False):
    fig = plt.figure(figsize=(14, 12))
    gs  = gridspec.GridSpec(4, 1, hspace=0.55)
    obs  = np.expm1(result.observed)  if escala_orig else result.observed
    trnd = np.expm1(result.trend)     if escala_orig else result.trend
    seas = result.seasonal.apply(np.exp) if escala_orig else result.seasonal
    res  = result.resid
    seas_label = 'Estacionalidad — Factor S (1.0=neutro)' if escala_orig else 'Estacionalidad (S)'
    panels = [
        (obs,  'Serie Original',     BLUE,   money_fmt if escala_orig else None),
        (trnd, 'Tendencia (T)',      RED,    money_fmt if escala_orig else None),
        (seas, seas_label,           GREEN,  None),
        (res,  'Residual / Ruido (R)', ORANGE, None),
    ]
    for i, (data, lbl, color, fmt) in enumerate(panels):
        ax = fig.add_subplot(gs[i])
        ax.plot(data.index, data.values, color=color, linewidth=1.5)
        ref = 1.0 if (i == 2 and escala_orig) else 0.0
        if i in (2, 3):
            ax.axhline(ref, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
        if fmt:
            ax.yaxis.set_major_formatter(fmt)
        ax.set_title(lbl, fontsize=11, fontweight='bold')
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.show()

plot_decomposition(decomp_cls,
    f'Descomposición Clásica — {model_type}',
    escala_orig=is_mult)

In [ ]:
# seasonal=53: debe ser impar y ≥ period+2
res_stl = STL(serie_work, period=52, seasonal=53, robust=True).fit()

trend_plot = np.expm1(res_stl.trend)           if is_mult else res_stl.trend
seas_plot  = res_stl.seasonal.apply(np.exp)    if is_mult else res_stl.seasonal
seas_ref   = 1.0                               if is_mult else 0.0
seas_lbl   = 'Estacionalidad — Factor S (1.0=neutro)' if is_mult else 'Estacionalidad (S)'

fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(4, 1, hspace=0.55)
panels = [
    (serie,       'Serie Original',      BLUE),
    (trend_plot,  'Tendencia (T)',        RED),
    (seas_plot,   seas_lbl,              GREEN),
    (res_stl.resid, 'Residual / Ruido (R)', ORANGE),
]
for i, (data, lbl, color) in enumerate(panels):
    ax = fig.add_subplot(gs[i])
    ax.plot(data.index, data.values, color=color, linewidth=1.5)
    if i in (0, 1):
        ax.yaxis.set_major_formatter(money_fmt)
    if i == 2:
        ax.axhline(seas_ref, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
    if i == 3:
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.set_title(lbl, fontsize=11, fontweight='bold')
fig.suptitle(f'Descomposición STL — {model_type} (LOESS, Robusta)',
             fontsize=14, fontweight='bold', y=1.01)
plt.show()

---
## 4. Análisis de componentes

In [ ]:
trend        = trend_plot.dropna()
trend_yearly = trend.resample('YE').mean()
growth       = trend_yearly.pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(trend.index, trend.values, color=RED, linewidth=2.5)
axes[0].yaxis.set_major_formatter(money_fmt)
axes[0].set_title('Componente de Tendencia', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Ingresos semanales')

bars = axes[1].bar(growth.index.year, growth.values,
                   color=[GREEN if v > 0 else RED for v in growth.values],
                   edgecolor='white', width=0.5)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Crecimiento Anual de la Tendencia (%)', fontsize=12, fontweight='bold')
for bar, val in zip(bars, growth.values):
    if not np.isnan(val):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)
fig.suptitle('Análisis de Tendencia', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Crecimiento anual de la tendencia:')
for yr, val in growth.items():
    if not np.isnan(val):
        print(f'  {yr.year}: {val:+.1f}%')

In [ ]:
seas_profile = (
    seas_plot
    .groupby(seas_plot.index.isocalendar().week)
    .mean()
)
# Uplift real: factor - 1 (multiplicativo) o valor absoluto (aditivo)
uplift = (seas_profile - 1) if is_mult else seas_profile
uplift_pct = uplift * 100 if is_mult else (seas_profile / serie.mean()) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

neutral = 1.0 if is_mult else 0.0
colors_bar = [RED if (v == seas_profile.max() or v == seas_profile.min())
              else GREEN if v > neutral else BLUE
              for v in seas_profile.values]
axes[0].bar(seas_profile.index, seas_profile.values, color=colors_bar, edgecolor='white', width=0.8)
axes[0].axhline(neutral, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title(
    'Perfil Estacional — Factor por Semana (1.0 = neutro)' if is_mult
    else 'Perfil Estacional — Efecto por Semana ($)',
    fontsize=12, fontweight='bold')
axes[0].set_xlabel('Semana del año')

for yr, color in zip(sorted(seas_plot.index.year.unique()), [BLUE, GREEN, ORANGE, PURPLE]):
    s_yr = seas_plot[seas_plot.index.year == yr]
    axes[1].plot(range(len(s_yr)), s_yr.values, color=color, linewidth=1.8, label=str(yr), alpha=0.85)
axes[1].axhline(neutral, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Estacionalidad por Año — ¿Estable o cambia?', fontsize=12, fontweight='bold')
axes[1].legend(title='Año')

fig.suptitle('Análisis de Estacionalidad', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 5 semanas (mayor uplift):')
for wk, pct in uplift_pct.nlargest(5).items():
    factor_str = f'factor {seas_profile[wk]:.3f}x → ' if is_mult else ''
    print(f'  Semana {int(wk):>2}: {factor_str}{pct:+.1f}% sobre promedio')

print('\nBottom 5 semanas (mayor caída):')
for wk, pct in uplift_pct.nsmallest(5).items():
    factor_str = f'factor {seas_profile[wk]:.3f}x → ' if is_mult else ''
    print(f'  Semana {int(wk):>2}: {factor_str}{pct:+.1f}% sobre promedio')

In [ ]:
resid     = res_stl.resid.dropna()
var_total = serie_work.var()
var_resid = resid.var()
r2        = 1 - (var_resid / var_total)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(resid.index, resid.values, color=ORANGE, linewidth=1, alpha=0.8)
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set_title('Residual en el Tiempo', fontsize=11, fontweight='bold')

axes[1].hist(resid, bins=40, color=ORANGE, edgecolor='white', alpha=0.85)
axes[1].axvline(resid.mean(), color=RED, linewidth=2, linestyle='--',
                label=f'Media: {resid.mean():.4f}')
axes[1].set_title('Distribución del Residual\n(idealmente centrado en 0)',
                  fontsize=11, fontweight='bold')
axes[1].legend()

sizes  = [r2 * 100, (1 - r2) * 100]
labels = ['T + S (explicado)', 'Residual\n(no explicado)']
axes[2].pie(sizes, labels=labels, colors=[GREEN, ORANGE],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[2].set_title('Varianza explicada', fontsize=11, fontweight='bold')

fig.suptitle('Análisis del Residual / Ruido', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

r2_status = 'Fuerte (≥0.80)' if r2 >= 0.8 else 'Usable (0.50–0.80)' if r2 >= 0.5 else 'Débil (<0.50)'
print(f'R² aproximado: {r2:.4f} → {r2_status}')

---
## 5. Validación formal del residual

Un buen R² no es suficiente. El residual debe ser **ruido blanco**: sin estructura, sin autocorrelación.  
Si no lo es → el modelo dejó información sobre la mesa → el forecast va a fallar.

| Test | H₀ | p > 0.05 |
|---|---|---|
| **Ljung-Box** | No hay autocorrelación | ✅ Ruido blanco |

> El análisis profundo de ACF/PACF viene en el **Video 5**. Aquí Ljung-Box es suficiente para validar la descomposición.

In [ ]:
lb = acorr_ljungbox(resid, lags=[10, 20, 30], return_df=True)

print('─' * 54)
print('  Test de Ljung-Box — Residual STL')
print('  H₀: No hay autocorrelación (ruido blanco)')
print('─' * 54)
print(f'  {"Rezago":>6}  {"Estadístico":>13}  {"p-valor":>9}  Resultado')
print('─' * 54)
lb_pass = True
lb_pval_20 = None
for lag, row in lb.iterrows():
    ok     = row['lb_pvalue'] > 0.05
    status = '✅ Ruido blanco' if ok else '❌ Hay estructura'
    if lag == 20:
        lb_pval_20 = row['lb_pvalue']
    if not ok:
        lb_pass = False
    print(f'  {lag:>6}  {row["lb_stat"]:>13.4f}  {row["lb_pvalue"]:>9.4f}  {status}')
print('─' * 54)

In [ ]:
# Reconstruir con T+S+R daría error ≈ 0 SIEMPRE, porque por construcción
# T + S + R = serie. Para que el MAPE signifique algo medimos cuánto explican
# la Tendencia y la Estacionalidad por sí solas; el resto es el residual R.
recon_ts_work = res_stl.trend + res_stl.seasonal
reconstructed = np.expm1(recon_ts_work) if is_mult else recon_ts_work
error_abs     = (serie - reconstructed).abs()
mape          = mean_absolute_percentage_error(serie, reconstructed) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(serie.index, serie.values, color=BLUE, linewidth=2, label='Original', alpha=0.7)
axes[0].plot(reconstructed.index, reconstructed.values,
             color=RED, linewidth=1.5, linestyle='--', label='Reconstruida (solo T+S)')
axes[0].yaxis.set_major_formatter(money_fmt)
axes[0].set_title('Original vs Reconstruida (T + S)', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].plot(error_abs.index, error_abs.values, color=ORANGE, linewidth=1)
axes[1].yaxis.set_major_formatter(money_fmt)
axes[1].set_title(f'Error de Reconstrucción | MAPE: {mape:.2f}%', fontsize=12, fontweight='bold')

fig.suptitle('Validación: ¿cuánto explican Tendencia + Estacionalidad?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

mape_status = 'Excelente (<5%)' if mape < 5 else 'Bueno (5–15%)' if mape < 15 else 'Revisar (>15%)'
print(f'MAPE con solo T+S: {mape:.4f}% → {mape_status}')
print('El error restante es el residual R (lo que T+S no explican), validado con Ljung-Box arriba.')

---
## 6. Selección automática de modelo: Clásico vs STL

No elegimos por preferencia — elegimos con datos.  
Menor varianza residual + residual es ruido blanco = mejor modelo.

In [ ]:
def evaluate_decomposition(series_work, series_orig, trend, seasonal, resid, label, is_mult):
    r2 = 1 - (np.var(resid) / np.var(series_work))
    # MAPE con SOLO T+S (sin residual): incluir R daría ≈0 por construcción
    recon = np.expm1(trend + seasonal) if is_mult else (trend + seasonal)
    recon_clean = recon.dropna()
    common_idx  = series_orig.index.intersection(recon_clean.index)
    mape = mean_absolute_percentage_error(series_orig.loc[common_idx], recon_clean.loc[common_idx]) * 100
    lb   = acorr_ljungbox(resid.dropna(), lags=[20], return_df=True)
    lb_p = lb['lb_pvalue'].iloc[0]
    valid = r2 >= 0.8 and lb_p > 0.05
    return {
        'modelo':        label,
        'r2':            round(r2, 4),
        'std_residual':  round(float(resid.std()), 6),
        'mape_ts_%':     round(mape, 4),
        'ljung_box_p20': round(lb_p, 4),
        'ruido_blanco':  '✅' if lb_p > 0.05 else '❌',
        'modelo_valido': '✅ VÁLIDO' if valid else '❌ REVISAR',
    }

eval_cls = evaluate_decomposition(
    serie_work, serie,
    decomp_cls.trend.dropna(), decomp_cls.seasonal,
    decomp_cls.resid.dropna(), 'Clásico', is_mult)

eval_stl = evaluate_decomposition(
    serie_work, serie,
    res_stl.trend, res_stl.seasonal,
    res_stl.resid, 'STL', is_mult)

comp_df = pd.DataFrame([eval_cls, eval_stl]).set_index('modelo')

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

axes[0].plot(serie.index, serie.values, color=BLUE, alpha=0.35, linewidth=1, label='Original')
cls_trend = np.expm1(decomp_cls.trend) if is_mult else decomp_cls.trend
stl_trend = np.expm1(res_stl.trend)    if is_mult else res_stl.trend
axes[0].plot(cls_trend.index, cls_trend.values,
             color=RED, linewidth=2.5, linestyle='--', label='Tendencia Clásica')
axes[0].plot(stl_trend.index, stl_trend.values,
             color=GREEN, linewidth=2.5, label='Tendencia STL')
axes[0].yaxis.set_major_formatter(money_fmt)
axes[0].set_title('Tendencias: Clásica vs STL', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].plot(decomp_cls.resid.dropna().index, decomp_cls.resid.dropna().values,
             color=RED, linewidth=1, alpha=0.7, linestyle='--', label='Residual Clásico')
axes[1].plot(res_stl.resid.index, res_stl.resid.values,
             color=GREEN, linewidth=1, alpha=0.7, label='Residual STL')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle=':')
axes[1].set_title('Residuales — Menor varianza = mejor ajuste', fontsize=12, fontweight='bold')
axes[1].legend()

fig.suptitle('Clásica vs STL', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

mejor = 'STL' if eval_stl['std_residual'] < eval_cls['std_residual'] else 'Clásico'
mejor_eval = eval_stl if mejor == 'STL' else eval_cls

print('\nComparación de modelos:')
print(comp_df.to_string())
print()
print('─' * 54)
print('  DECISIÓN AUTOMÁTICA')
print('─' * 54)
print(f'  Modelo elegido  : {mejor}')
print(f'  Tipo            : {model_type}')
print(f'  R²              : {mejor_eval["r2"]:.4f}')
print(f'  MAPE T+S        : {mejor_eval["mape_ts_%"]:.4f}%')
print(f'  Ljung-Box p20   : {mejor_eval["ljung_box_p20"]:.4f}')
print(f'  Ruido blanco    : {mejor_eval["ruido_blanco"]}')
print(f'  Estado          : {mejor_eval["modelo_valido"]}')
print('─' * 54)

modelo_valido = mejor_eval['modelo_valido'] == '✅ VÁLIDO'

---
## 7. Pipeline completo reutilizable

Todo el flujo encapsulado en una función.  
Pásale cualquier serie semanal y devuelve componentes + métricas + decisión.

In [ ]:
def full_decomposition_pipeline(series: pd.Series, period: int = 52, verbose: bool = True) -> dict:
    """
    Pipeline completo de descomposición de series de tiempo.
    1. Detecta tipo (aditivo/multiplicativo) por corr(media móvil, std móvil)
    2. Transforma si es necesario (log1p)
    3. Descompone con STL
    4. Valida residual (Ljung-Box + MAPE con solo T+S)
    5. Devuelve componentes + métricas + flag de validez
    """
    # 1. Detección de tipo (señal robusta: la amplitud escala con el nivel)
    rmean = series.rolling(12).mean()
    rstd  = series.rolling(12).std()
    corr_ms = pd.concat([rmean, rstd], axis=1).dropna().corr().iloc[0, 1]
    mult = corr_ms > 0.6

    s_work = np.log1p(series) if mult else series

    # 3. STL (seasonal debe ser impar)
    res = STL(s_work, period=period, seasonal=period + 1, robust=True).fit()

    # 4. Reconstrucción con solo T+S (incluir R daría MAPE ≈ 0 por construcción)
    recon = np.expm1(res.trend + res.seasonal) if mult else (res.trend + res.seasonal)
    resid = res.resid.dropna()
    r2    = 1 - (np.var(resid) / np.var(s_work))
    common = series.index.intersection(recon.dropna().index)
    mape  = mean_absolute_percentage_error(series.loc[common], recon.dropna().loc[common]) * 100
    lb    = acorr_ljungbox(resid, lags=[20], return_df=True)
    lb_p  = lb['lb_pvalue'].iloc[0]
    valid = r2 >= 0.8 and lb_p > 0.05

    if verbose:
        tag = 'MULTIPLICATIVO' if mult else 'ADITIVO'
        print(f'  corr(media, std): {corr_ms:+.3f}')
        print(f'  Tipo de modelo  : {tag}')
        print(f'  Transformación  : {"log1p" if mult else "ninguna"}')
        print(f'  R²              : {r2:.4f}')
        print(f'  MAPE T+S        : {mape:.4f}%')
        print(f'  Ljung-Box p20   : {lb_p:.4f}')
        print(f'  Modelo válido   : {"✅ SÍ" if valid else "❌ REVISAR"}')

    return {
        'is_multiplicative': mult,
        'stl_result':        res,
        'trend':             np.expm1(res.trend)          if mult else res.trend,
        'seasonal_factor':   res.seasonal.apply(np.exp)   if mult else res.seasonal,
        'resid':             resid,
        'reconstructed':     recon,
        'r2':                round(r2, 4),
        'mape':              round(mape, 4),
        'lb_pvalue_20':      round(lb_p, 4),
        'modelo_valido':     valid,
    }

print('─' * 48)
print('  PIPELINE: Serie total semanal')
print('─' * 48)
results = full_decomposition_pipeline(serie)
print('─' * 48)

In [ ]:
# Elegimos por revenue: una serie densa y regular (tier A). La descomposición STL
# no es apropiada para slow-movers intermitentes — esos van por otra ruta (Video 7).
sku_target = df.groupby('sku_id')['revenue'].sum().idxmax()
serie_sku  = (
    df[df['sku_id'] == sku_target]
    .groupby('date')['revenue']
    .sum()
    .resample('W-MON')
    .sum()
)
print(f'─' * 48)
print(f'  PIPELINE: {sku_target} (mayor revenue)')
print(f'─' * 48)
results_sku = full_decomposition_pipeline(serie_sku)
print(f'─' * 48)

---
## 8. Puente al Video 5 — ACF y PACF

Con los componentes aislados, puedes:
- **Desestacionalizar** la serie para modelar solo tendencia + ruido
- **Re-estacionalizar** el forecast al final multiplicando por el factor estacional

Pero antes de modelar, necesitamos entender la **memoria** de la serie desestacionalizada:  
¿cuántos rezagos pasados predicen el presente? Eso es lo que responden ACF y PACF en el siguiente video.

In [ ]:
# Para el caso multiplicativo la hacemos EXACTA en la escala log1p:
#   log1p(serie) = T + S + R  →  quitar S y volver con expm1
res = results['stl_result']
if results['is_multiplicative']:
    deseasonalized   = np.expm1(np.log1p(serie) - res.seasonal)
    seas_factor_full = results['seasonal_factor']          # factor ≈ ×, solo para graficar
else:
    deseasonalized   = serie - res.seasonal
    seas_factor_full = res.seasonal

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(serie.index, serie.values, color=BLUE, linewidth=1.2, alpha=0.7, label='Original')
axes[0].plot(deseasonalized.index, deseasonalized.values,
             color=RED, linewidth=1.8, label='Desestacionalizada')
axes[0].yaxis.set_major_formatter(money_fmt)
axes[0].set_title('Original vs Desestacionalizada', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].plot(seas_factor_full.index, seas_factor_full.values, color=GREEN, linewidth=1.5)
ref = 1.0 if results['is_multiplicative'] else 0.0
axes[1].axhline(ref, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title(
    'Factor Estacional extraído (1.0 = neutro)' if results['is_multiplicative']
    else 'Componente Estacional extraído',
    fontsize=12, fontweight='bold')

fig.suptitle('Puente hacia el Forecasting: Serie Desestacionalizada',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Flujo de forecasting con descomposición:')
print('  1. Desestacionalizar la serie                → modelar tendencia + ruido')
print('  2. Aplicar modelo (ARIMA / Prophet / XGBoost) sobre la serie limpia')
print('  3. Re-estacionalizar: forecast × factor_estacional')
print('\n→ Videos siguientes: cada modelo aplicará este flujo sobre esta base.')

---
### Próximo video
**Video 5 — ACF y PACF: identificando el orden de un modelo ARIMA**  
Veremos cómo leer los correlogramas para identificar los parámetros `p` y `q`,  
y por qué el Ljung-Box que vimos aquí es solo la versión comprimida de un análisis mucho más rico.